# Figure 5 — Coverage calibration (B&W publication PDF)

Reads `article/results/coverage_simulation.csv` and renders a print-quality PDF figure showing empirical vs nominal 95% coverage for each CI method as sample size grows. Methods are distinguished by marker shape *and* line style so the figure remains readable when photocopied or printed in grayscale.

**Output:** `article/figures/fig5_coverage_calibration.pdf` (vector, 1200 DPI raster fallback).


In [ ]:
from __future__ import annotations
import sys
from pathlib import Path
import pandas as pd

REPO = Path.cwd()
while not (REPO / 'pyproject.toml').exists() and REPO != REPO.parent:
    REPO = REPO.parent
if str(REPO / 'notebooks' / 'article' / 'figures') not in sys.path:
    sys.path.insert(0, str(REPO / 'notebooks' / 'article' / 'figures'))
from _style import METHOD_STYLE, METHOD_LABEL, DOUBLE_COL_INCHES, apply_rc

import matplotlib.pyplot as plt
apply_rc()

CSV = REPO / 'article' / 'results' / 'coverage_simulation.csv'
PDF = REPO / 'article' / 'figures' / 'fig5_coverage_calibration.pdf'
PDF.parent.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(CSV)
print(df['method'].unique())

In [ ]:
fig, ax = plt.subplots(figsize=(DOUBLE_COL_INCHES * 0.7, 3.0))
for method in METHOD_STYLE:
    sub = df[df['method'] == method].sort_values('n')
    if sub.empty:
        continue
    style = METHOD_STYLE[method]
    ax.plot(
        sub['n'], sub['empirical_coverage'],
        marker=style['marker'], linestyle=style['linestyle'],
        markerfacecolor=style['face'], markeredgecolor=style['edge'],
        color='black', linewidth=1.0, markersize=5,
        label=METHOD_LABEL[method],
    )
ax.axhline(0.95, color='gray', linestyle='-', linewidth=0.7, alpha=0.7)
ax.set_xscale('log')
ax.set_xlabel('Sample size $n$ (log scale)')
ax.set_ylabel('Empirical 95\\% coverage')
ax.set_ylim(0.88, 1.00)
ax.text(df['n'].max(), 0.951, 'nominal 0.95', fontsize=7, color='gray', va='bottom', ha='right')
ax.legend(loc='lower right', frameon=False, ncol=2)
fig.tight_layout()
fig.savefig(PDF, format='pdf', bbox_inches='tight')
print('wrote', PDF.relative_to(REPO))
plt.show()